# GigShield Fraud Model — Training Notebook

Mirror of `train/train.py` with plots. Run `python -m train.generate_synthetic` first to build the parquet.

This notebook was generated programmatically. To re-run from the CLI instead, use:
```bash
python -m train.generate_synthetic
python -m train.train
```

In [ ]:
import sys, os
from pathlib import Path
# Ensure the package root is importable when running from train/
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix

from app.features import FEATURE_NAMES

In [ ]:
df = pd.read_parquet(Path('..') / 'models' / 'synthetic_train.parquet')
print(df.shape)
df.head()

In [ ]:
df['label'].value_counts().plot(kind='bar')
plt.title('Class balance')
plt.show()

In [ ]:
train_df = df[df.split == 'train']
val_df   = df[df.split == 'val']
test_df  = df[df.split == 'test']
X_train, y_train = train_df[FEATURE_NAMES].values, train_df['label'].values
X_val, y_val     = val_df[FEATURE_NAMES].values,   val_df['label'].values
X_test, y_test   = test_df[FEATURE_NAMES].values,  test_df['label'].values

In [ ]:
clf = lgb.LGBMClassifier(
    objective='binary', metric='auc', learning_rate=0.05, num_leaves=31,
    min_data_in_leaf=100, feature_fraction=0.8, bagging_fraction=0.8,
    bagging_freq=5, n_estimators=500, seed=42,
)
clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], eval_metric='auc',
        callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)])

In [ ]:
cal = CalibratedClassifierCV(estimator=clf, method='sigmoid', cv='prefit')
cal.fit(X_val, y_val)
y_score = cal.predict_proba(X_test)[:, 1]
print('AUC   =', roc_auc_score(y_test, y_score))
print('PR-AUC=', average_precision_score(y_test, y_score))

In [ ]:
p, r, _ = precision_recall_curve(y_test, y_score)
plt.plot(r, p)
plt.xlabel('Recall'); plt.ylabel('Precision'); plt.title('PR curve (test)')
plt.grid(True); plt.show()

In [ ]:
lgb.plot_importance(clf, max_num_features=15, importance_type='gain', figsize=(8, 6))
plt.tight_layout(); plt.show()

In [ ]:
import shap
explainer = shap.TreeExplainer(clf.booster_)
sv = explainer.shap_values(X_test[:500])
if isinstance(sv, list): sv = sv[1]
shap.summary_plot(sv, X_test[:500], feature_names=FEATURE_NAMES)